<a href="https://colab.research.google.com/github/yunussfr/MSIDdetection/blob/main/RelayInceptionV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# ==========================================
# 1. VERİ SETİ HAZIRLIĞI VE DÜZENLİ AYIRMA
# ==========================================
dataset_path = '/content/drive/MyDrive/Monkeypox Skin Image Dataset' # Kendi yolunu gir. Alt klasörleri barındıran ana dizin olmalı.
img_size = (299, 299)
batch_size = 32

# validation_split=0.2: Her klasörden (sınıftan) orantılı olarak %20 veri ayırır (Stratified).
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False # Metriklerin doğru hesaplanması için test setinde shuffle=False OLMALI
)

# ==========================================
# 2. DENGESİZ VERİ İÇİN CEZA (AĞIRLIK) SİSTEMİ
# ==========================================
# Az olan sınıfa yüksek, çok olan sınıfa düşük ceza puanı (ağırlık) atar.
train_classes = train_generator.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_classes),
    y=train_classes
)
# Keras'ın anlayacağı sözlük (dictionary) formatına çeviriyoruz: {0: 1.5, 1: 0.8, ...}
class_weight_dict = dict(enumerate(class_weights))
print("\n--- Sınıf Ağırlıkları (Ceza Puanları) ---")
for class_index, weight in class_weight_dict.items():
    class_name = list(train_generator.class_indices.keys())[class_index]
    print(f"{class_name} (Sınıf {class_index}): {weight:.4f}")

# ==========================================
# 3. HER EPOCH SONU METRİK GÖSTERİCİ & EARLY STOPPING
# ==========================================
class EpochMetricsAndEarlyStopping(Callback):
    def __init__(self, validation_generator, patience=3):
        super(EpochMetricsAndEarlyStopping, self).__init__()
        self.val_gen = validation_generator
        self.patience = patience
        self.best_f1 = -1.0
        self.wait = 0
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        self.val_gen.reset()
        y_true = self.val_gen.classes

        # Sadece tahmin yapıyoruz, verbose=0 ile log kirliliğini önlüyoruz
        y_pred_prob = self.model.predict(self.val_gen, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)

        # Metrikleri hesapla (Macro average dengesiz veri setlerinde en güvenilir olanıdır)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        print(f"\n[Epoch {epoch+1} Sonuçları] -> "
              f"Accuracy: {acc:.4f} | "
              f"Precision: {precision:.4f} | "
              f"Recall: {recall:.4f} | "
              f"F1 Macro: {f1:.4f}")

        # En iyi modeli kaydetme ve erken durdurma kontrolü
        if f1 > self.best_f1:
            self.best_f1 = f1
            self.wait = 0
            self.best_weights = self.model.get_weights()
            print(f">>> F1 Macro arttı! Yeni rekor: {self.best_f1:.4f}. Ağırlıklar hafızaya alındı.")
        else:
            self.wait += 1
            print(f">>> Düşüş / İlerleme yok. Sabır: {self.wait} / {self.patience}")
            if self.wait >= self.patience:
                print(f"\n[DURDURULDU] Model ezberlemeye başladı veya gelişmiyor. ({self.patience} epoch limitine ulaşıldı)")
                self.model.stop_training = True
                print("[GERİ YÜKLEME] En iyi F1 skoruna sahip ağırlıklar sisteme geri yükleniyor...")
                self.model.set_weights(self.best_weights)
        print("="*60)

# ==========================================
# 4. INCEPTION V3 MODEL MİMARİSİ
# ==========================================
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False # Transfer Learning gereği önceden öğrenilenleri koruyoruz

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
num_classes = train_generator.num_classes
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# ==========================================
# 5. EĞİTİMİ BAŞLAT
# ==========================================
max_epochs = 30
custom_callback = EpochMetricsAndEarlyStopping(validation_generator=validation_generator, patience=3)

print("\nEğitim Başlıyor...\n" + "="*60)
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=max_epochs,
    class_weight=class_weight_dict, # <<< Dengesizlik cezası (Ağırlıklar) modele buradan verilir
    callbacks=[custom_callback]     # <<< Her epoch sonu özel metrikleri ve durdurmayı bu halleder
)

print("\nSistem Eğitimi Tamamlandı! Model en verimli halinde.")

Found 618 images belonging to 4 classes.
Found 152 images belonging to 4 classes.

--- Sınıf Ağırlıkları (Ceza Puanları) ---
Chickenpox (Sınıf 0): 1.7965
Measles (Sınıf 1): 2.1164
Monkeypox (Sınıf 2): 0.6897
Normal (Sınıf 3): 0.6574
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Eğitim Başlıyor...
Epoch 1/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.4125 - loss: 1.5145
[Epoch 1 Sonuçları] -> Accuracy: 0.7434 | Precision: 0.7192 | Recall: 0.7802 | F1 Macro: 0.7136
>>> F1 Macro arttı! Yeni rekor: 0.7136. Ağırlıklar hafızaya alındı.
20/20 ━━━━━━━━━━━━━━━━━━━━ 321s 15s/step - accuracy: 0.5146 - loss: 1.2921 - val_accuracy: 0.7105 - val_loss: 0.7375
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 983ms/step - accuracy: 0.7210 - loss: 0.7707
[Epoch 2 Sonuçları] -> Accuracy: 0.7763 | Precision: 0.7216 | Recall: 0.7375 | F1 Macro: 0.7214
>>> F1 Macro arttı! Yeni rekor: 0.7214. Ağırlıklar hafızaya alındı.
20/20 ━━━━━━━━━━━━━━━━━━━━ 32s 2s/step - accuracy: 0.7265 - loss: 0.7431 - val_ac